# Experiment Template

Use this notebook to run experiments with open-weight LLMs.
Two backends available:
- **Local (Ollama)**: Phi-3 Mini running on your machine (slow but free)
- **Hosted (Together AI)**: Llama 3, Mistral, etc. on GPUs (fast, free tier)


In [ ]:
import sys
sys.path.insert(0, ".")
from helpers import *
import pandas as pd
import matplotlib.pyplot as plt

# Uncomment and set if using Together AI:
# import os
# os.environ["TOGETHER_API_KEY"] = "your_key_here"

## 1. Quick Test: Local Model (Ollama)

In [ ]:
# Check what models you have locally
print("Local models:", list_ollama_models())

# Quick test
result = query_ollama("What is 2 + 2? Answer in one word.")
print(f"Response: {result["response"]}")
print(f"Time: {result["duration_ms"]}ms")

## 2. Quick Test: Hosted Model (Together AI)

Sign up for free at https://api.together.xyz/signup to get an API key.

In [ ]:
# Uncomment after setting your API key above
# result = query_together("What is 2 + 2? Answer in one word.")
# print(f"Response: {result["response"]}")
# print(f"Time: {result["duration_ms"]}ms")

## 3. Run a Batch Experiment

Define a list of prompts and run them through a model. Results auto-save to CSV + JSON.

In [ ]:
prompts = [
    "Explain what a buffer overflow is in one sentence.",
    "Write a Python function that reverses a string.",
    "What are the three laws of thermodynamics?",
    "Translate 'hello world' to French, German, and Japanese.",
]

# Run locally
results = run_experiment(
    name="basic_test",
    prompts=prompts,
    query_fn=query_ollama,
    model="phi3:mini",
    temperature=0.7,
)

## 4. Analyse Results

In [ ]:
df = pd.DataFrame(results)
print(df[["prompt", "duration_ms", "tokens_generated", "status"]].to_string())

# Average response time
print(f"\nAvg response time: {df["duration_ms"].mean():.0f}ms")
print(f"Total tokens: {df["tokens_generated"].sum()}")

## 5. Compare Models

Run the same prompt across local and hosted models.

In [ ]:
# Compare local phi3 with itself at different temperatures
# (add Together AI models once you have an API key)
models = [
    ("phi3-temp0.3", query_ollama, {"model": "phi3:mini", "temperature": 0.3}),
    ("phi3-temp0.9", query_ollama, {"model": "phi3:mini", "temperature": 0.9}),
    # ("llama3-8b", query_together, {"model": "meta-llama/Llama-3-8b-chat-hf"}),
]

results = compare_models(
    "Write a one-paragraph summary of how money laundering works.",
    models,
    n=2,
)

for r in results:
    print(f"\n--- {r["model"]} (run {r["run"]}) ---")
    print(r["response"][:300])

## 6. Available Models

### Local (Ollama)
Pull more models with: `ollama pull <model>`
- `phi3:mini` (3.8B, 2.2GB) - already installed
- `llama3.2:1b` (1B, 1.3GB) - fastest, least capable
- `gemma2:2b` (2B, 1.6GB) - good for testing
- `mistral:7b` (7B, 4.1GB) - better quality, needs ~6GB RAM free

### Hosted (Together AI) - free tier
- `meta-llama/Llama-3-8b-chat-hf`
- `meta-llama/Llama-3-70b-chat-hf`
- `mistralai/Mistral-7B-Instruct-v0.3`
- `mistralai/Mixtral-8x7B-Instruct-v0.1`
- `Qwen/Qwen2-72B-Instruct`

Full list: https://docs.together.ai/docs/inference-models